# 🧠 Cómo Funciona Mi Sistema — Guía Técnica de CASSERISISSIMA 2.0

**Autores**: Br. Jorfran Gil · Br. Yefferson Hernández  
**Universidad de Oriente — Núcleo de Monagas**

---

Este notebook es una guía para entender cómo funciona cada parte del sistema CASSERISISSIMA 2.0. Está escrito de forma didáctica para que cualquier persona del equipo pueda entender la lógica del proyecto.

## 1. Mapa del Proyecto — ¿Qué hace cada archivo?

```
CASSERISISSIMA-2.0/
│
├── src/                          ← TODO el código Python vive aquí
│   ├── main.py                   ← PUNTO DE ENTRADA: arranca la API
│   │
│   ├── core/ml/                  ← EL CEREBRO: Machine Learning
│   │   ├── feature_engineering.py  ← Transforma ventas crudas en features
│   │   ├── pipeline.py             ← Construye el pipeline de scikit-learn
│   │   ├── model_trainer.py        ← Entrena y elige el mejor modelo
│   │   ├── model_registry.py       ← Guarda versiones de los modelos
│   │   └── benchmark.py            ← Compara métricas de todos los productos
│   │
│   ├── core/operations_research/  ← LAS DECISIONES: Inventario
│   │   ├── newsvendor.py           ← ¿Cuántas tortas hornear?
│   │   └── reorder_point.py        ← ¿Cuándo pedir insumos?
│   │
│   ├── db/                        ← LA MEMORIA: Base de datos
│   │   ├── database.py             ← Conexión a SQLite
│   │   ├── models.py               ← Estructura de las tablas
│   │   └── seed.py                 ← Carga los datos del CSV a la DB
│   │
│   └── routers/                   ← LOS ENDPOINTS: API REST
│       ├── dashboard.py            ← Estadísticas generales
│       ├── sales.py                ← Registro de ventas
│       ├── predictions.py          ← Pronósticos de IA
│       ├── insights.py             ← Alertas automáticas
│       └── scenarios.py            ← Cambio de escenarios
│
├── frontend/                      ← LA CARA: Interfaz de usuario
│   ├── app/                        ← Páginas (Dashboard, Predicciones, Ventas)
│   ├── components/                 ← Componentes visuales reutilizables
│   └── lib/                        ← Conexión con la API del backend
│
└── data/                          ← LOS DATOS
    ├── raw/                        ← CSVs originales de la pastelería
    └── models/                     ← Modelos entrenados (.joblib)
```

## 2. ¿Cómo fluye una predicción?

Cuando el usuario abre la página de predicciones en el navegador, esto es lo que pasa:

### Paso a paso

1. **El usuario** hace clic en un producto en el Dashboard
2. **El Frontend** (Next.js) envía una petición HTTP al Backend:
   ```
   GET http://localhost:8000/api/v1/predictions/TL-CLASICA?horizon=14
   ```
3. **El Router** (`predictions.py`) recibe la petición y:
   - Busca el producto en la base de datos
   - Carga su historial de ventas del escenario activo
4. **Feature Engineering** transforma las ventas crudas en 40+ features
5. **El Model Trainer** entrena (o carga) el modelo para ese producto
6. **El Pipeline** genera la predicción + intervalos de confianza
7. **Newsvendor** calcula cuántas tortas hornear
8. **ROP** calcula cuándo pedir insumos
9. **El Router** empaqueta todo en un JSON y lo devuelve
10. **El Frontend** renderiza los gráficos de pronóstico

## 3. Feature Engineering — Explicación Simple

### ¿Qué son las features?

El modelo de Machine Learning no entiende "tortas" ni "fechas". Solo entiende **números**. La ingeniería de features convierte la información del negocio en números que el modelo puede procesar.

### Cada feature explicada con analogías

| Feature | Analogía | Ejemplo |
|---------|----------|---------|
| `lag_1` | "¿Cuánto vendí ayer?" | Si ayer vendí 3, es probable que hoy venda algo similar |
| `lag_7` | "¿Cuánto vendí el mismo día la semana pasada?" | Si el martes pasado vendí 5, este martes podría ser parecido |
| `rolling_mean_7` | "¿Cómo me fue esta semana en promedio?" | La tendencia reciente de 7 días |
| `rolling_std_7` | "¿Qué tan impredecible fue mi semana?" | Si la desviación es alta, las ventas variaron mucho |
| `ewm_7` | "¿Para dónde va la tendencia?" | Promedio que pesa más los días recientes |
| `is_weekend` | "¿Es fin de semana?" | Los sábados y domingos pueden tener patrones diferentes |
| `is_payday` | "¿Es quincena?" | Días 14-15 y 28-31 la gente tiene más dinero en Venezuela |
| `dow_sin/cos` | "¿Qué día de la semana es?" (en círculo) | El modelo entiende que domingo y lunes están "cerca" |
| `weekend_x_mean7` | "¿Es fin de semana Y hay tendencia alta?" | Efecto combinado de dos factores |

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

from db.database import SessionLocal, init_db
from db.models import Product, SaleTransaction

init_db()
db = SessionLocal()

# Cargar un producto de ejemplo
product = db.query(Product).filter(Product.is_active == True).first()

sales = (
    db.query(SaleTransaction.sale_date, SaleTransaction.quantity_sold)
    .filter(SaleTransaction.scenario_id == 2, SaleTransaction.product_id == product.id)
    .order_by(SaleTransaction.sale_date)
    .all()
)

sales_df = pd.DataFrame(
    [(r.sale_date, float(r.quantity_sold)) for r in sales],
    columns=['sale_date', 'quantity_sold']
)

print(f'Producto: {product.name} ({product.sku})')
print(f'Registros: {len(sales_df)}')

In [ ]:
from core.ml.feature_engineering import build_features, FEATURE_COLUMNS

# Construir features
df = build_features(sales_df, shelf_life_days=product.shelf_life_days)

# Mostrar un ejemplo: los últimos 5 días
print('Últimos 5 días con sus features principales:')
print()
key_features = ['quantity_sold', 'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_std_7',
                'is_weekend', 'is_payday', 'day_of_week']
df[key_features].tail(5).round(2)

## 4. Random Forest — Explicación Intuitiva

### ¿Qué es un Random Forest?

Imaginate que querés saber cuántas tortas Tres Leches vender mañana. En lugar de preguntarle a UNA sola persona, le preguntás a **200 expertos** (árboles de decisión). Cada experto:

1. Mira un **subconjunto aleatorio** de los datos históricos
2. Considera un **subconjunto aleatorio** de las features
3. Da su predicción independiente

La predicción final es el **promedio** de las 200 opiniones.

### ¿Por qué funciona?

- Si un experto se equivoca mucho, los otros 199 lo compensan
- La aleatorización evita que todos cometan el mismo error
- Es como la "sabiduría de las multitudes"

### ¿Cómo se calculan los intervalos de confianza?

Como cada árbol da una predicción diferente, podemos ver qué tan de acuerdo están:

- Si los 200 árboles dicen entre 2 y 3 tortas → **alta confianza**
- Si algunos dicen 1 y otros dicen 8 → **baja confianza** (bandas anchas)

El percentil 5 de las 200 predicciones es el límite inferior, y el percentil 95 es el superior.

In [ ]:
import joblib
from core.ml.model_trainer import train_product_model

# Entrenar el modelo
train_input = sales_df.copy()
train_input['sale_date'] = pd.to_datetime(train_input['sale_date']).dt.strftime('%Y-%m-%d')

result = train_product_model(
    sales_df=train_input,
    product_id=product.id,
    sku=product.sku,
    shelf_life_days=product.shelf_life_days,
)

# Cargar el modelo y ver los árboles individuales
pipeline = joblib.load(result['model_path'])
model = pipeline.named_steps['model']

if hasattr(model, 'estimators_'):
    print(f'Número de árboles: {len(model.estimators_)}')
    
    # Predecir con el último día
    X_last = df[FEATURE_COLUMNS].tail(1)
    X_transformed = pipeline[:-1].transform(X_last)
    
    tree_preds = [tree.predict(X_transformed)[0] for tree in model.estimators_]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(tree_preds, bins=30, color='#4ECDC4', edgecolor='white', alpha=0.8)
    ax.axvline(np.mean(tree_preds), color='red', linewidth=2, label=f'Media: {np.mean(tree_preds):.2f}')
    ax.axvline(np.percentile(tree_preds, 5), color='orange', linewidth=2, linestyle='--', label=f'P5: {np.percentile(tree_preds, 5):.2f}')
    ax.axvline(np.percentile(tree_preds, 95), color='orange', linewidth=2, linestyle='--', label=f'P95: {np.percentile(tree_preds, 95):.2f}')
    ax.set_title(f'Distribución de predicciones de {len(model.estimators_)} árboles — {product.name}', fontweight='bold')
    ax.set_xlabel('Predicción (unidades)')
    ax.set_ylabel('Número de árboles')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('El modelo ganador fue LightGBM (no tiene árboles individuales accesibles para este gráfico)')

## 5. Newsvendor — El Problema del Vendedor de Periódicos (con Tortas)

### La analogía

Imaginate que sos un vendedor de periódicos en los años 50:

- Cada mañana comprás N periódicos al por mayor (costo: $0.50 c/u)
- Los vendés a $1.00 c/u
- Los que NO vendiste se tiran a la basura al final del día

**¿Cuántos periódicos comprás?**

- Si comprás de más → pierdes $0.50 por cada periódico no vendido
- Si comprás de menos → perdés $0.50 de ganancia por cada venta perdida

### Aplicado a Casseríssimas

Cambiá "periódicos" por "tortas":

- **Torta Tres Leches**: Precio $15, Costo $5
- Si horneo de más: pierdo $5 por torta (se vence en 3 días)
- Si horneo de menos: pierdo $10 de ganancia (un cliente se fue sin torta)

### La fórmula

$$CR = \frac{\text{Precio} - \text{Costo}}{\text{Precio}} = \frac{15 - 5}{15} = 0.667$$

El CR de 0.667 dice: "conviene arriesgarse a producir un poco de más, porque la ganancia por venta ($10) es mayor que la pérdida por merma ($5)".

Luego se usa la distribución de la demanda predicha por el modelo para encontrar Q* (la cantidad óptima).

In [ ]:
from core.operations_research.newsvendor import calculate_critical_ratio, newsvendor_optimal_quantity

cr = calculate_critical_ratio(unit_cost=product.unit_cost, selling_price=product.selling_price)

print(f'=== Ejemplo con {product.name} ===')
print(f'Precio de venta:  ${product.selling_price:.2f}')
print(f'Costo unitario:   ${product.unit_cost:.2f}')
print(f'Ganancia por torta vendida:    ${product.selling_price - product.unit_cost:.2f}')
print(f'Pérdida por torta no vendida:  ${product.unit_cost:.2f}')
print(f'\nRatio Crítico (CR): {cr:.4f}')
print(f'\nInterpretación: El CR de {cr:.2f} significa que conviene')
if cr > 0.5:
    print(f'producir un poco de MÁS porque la ganancia ({product.selling_price - product.unit_cost:.0f})'
          f' supera la pérdida por merma ({product.unit_cost:.0f}).')
else:
    print(f'ser CONSERVADOR porque la pérdida por merma es alta.')

## 6. Glosario de Términos

| Término | Qué es | Analogía simple |
|---------|--------|------------------|
| **MAPE** | Error porcentual promedio del modelo | "El modelo se equivoca un X% en promedio" |
| **RMSE** | Error cuadrático (penaliza errores grandes) | "Cuánto duele cuando me equivoco por mucho" |
| **MAE** | Error absoluto en unidades físicas | "Me equivoco por 0.3 tortas al día" |
| **Lag** | Valor de la demanda en un punto pasado | "Cuánto vendí hace N días" |
| **Rolling Mean** | Promedio móvil de los últimos N días | "La tendencia reciente" |
| **Rolling Std** | Desviación estándar móvil | "Qué tan impredecible fue la semana" |
| **EWM** | Media Exponencialmente Ponderada | "Promedio que pesa más lo reciente" |
| **Winsorización** | Recortar valores extremos | "Ignorar el pedido de 50 tortas para una fiesta" |
| **IQR** | Rango Intercuartílico | "La variación normal de las ventas" |
| **TimeSeriesSplit** | Validación cruzada temporal | "Probar el modelo simulando el futuro" |
| **Feature Importance** | Importancia de cada variable para el modelo | "¿Qué dato le importa más al modelo?" |
| **Intervalo de Confianza** | Rango probable de la predicción | "Estoy 90% seguro de que será entre X e Y" |
| **Newsvendor** | Modelo de cantidad óptima de producción | "¿Cuántas tortas hornear?" |
| **ROP** | Punto de reorden de insumos | "¿Cuándo pedir más harina?" |
| **Critical Ratio** | Relación ganancia/pérdida del Newsvendor | "¿Vale la pena arriesgarse a producir más?" |
| **Safety Stock** | Stock de seguridad ante incertidumbre | "El colchón extra por si acaso" |
| **Lead Time** | Días que tarda el proveedor en entregar | "Cuánto tarda en llegar la harina" |
| **Pipeline (sklearn)** | Cadena de pasos de procesamiento | "La receta paso a paso del modelo" |
| **Joblib** | Formato para guardar modelos entrenados | "Guardar el cerebro del modelo en disco" |

In [ ]:
db.close()
print('✓ Sesión cerrada. ¡Listo para entender el sistema!')

---

*Guía técnica del sistema CASSERISISSIMA 2.0 — Trabajo Especial de Grado, Universidad de Oriente, Núcleo de Monagas, 2026*